# Lab 02 — Predict, measure, explain an inference bottleneck

**Outcome:** follow one exact prompt through token IDs, actual generated output and KV state; then produce a reproducible phase experiment, a matched batch comparison and two focused traces. Explain one prediction that did not survive measurement. This is not an HTTP benchmark or a quality test.

Read [Book 02](https://levshaazz.github.io/llm-serving-mastery/en/book/02/) and the [student guide](https://levshaazz.github.io/llm-serving-mastery/en/start/). You independently arrange a CUDA GPU (Linux/Colab/WSL; 16 GiB recommended). CPU contract checks do not replace the GPU run. Use your private repository's GitHub Issue with @levshaazz for diagnostics; do not post secrets or private artifacts publicly.

**Protocol v2:** two pinned models × five cases × five raw trials = **50 trials**. Cases are batch 1 × input {128,4096} × output {32,128}, plus batch 4/input 128/output 32. Batch 1/input 128/output 32 is the matched batch control. Output 512 and batch 8 are optional follow-ups only, after the core passes. Historical v1 results are not a v2 baseline.

**Before execution:** write three predictions: which phase changes with prompt length, which with output length, and whether batch 4 raises aggregate rate while changing per-sequence cadence. Name the measurement that could disprove each prediction.

Run one notebook/process at a time. Models reuse a persistent cache; execution artifacts get new directories. Stop on OOM, resource growth or a gate failure; preserve the partial JSON. Do not restart the entire matrix blindly.


## 0. See the exact prompt first — recorded replay, CPU only

Run the next two cells without installing anything or connecting a GPU. The actual recorded input and generated sequence are embedded in this notebook; they require only Python's standard library, no network request and no model download. This is **recorded instructor evidence**, not a GPU run you performed and not completion of the required lab.

**Predict before revealing:** the user asks “Continue the sequence with three numbers: 2, 4, 6,”. What else reaches the model? Does the first forward process one token or the entire prompt? After selecting its first output, has that new token already acquired K/V?

The natural-text replay uses a pinned 1.5B model and a maximum of 24 greedy output tokens, stopping at EOS. It synchronizes each step for inspection. The later required matrix uses synthetic repeated IDs, fixed output lengths and whole-phase timing. **Never attach the replay's natural prompt to the matrix's synthetic timing numbers.**


In [ ]:
import json
RECORDED_PROMPT = json.loads("{\"protocol\":\"topic02-concrete-prompt-v1\",\"scope\":\"one instrumented natural-EOS request; not a benchmark\",\"model\":{\"id\":\"Qwen/Qwen2.5-1.5B-Instruct\",\"revision\":\"989aa7980e4cf806f80c7fef2b1adb7bc71aa306\",\"kv_bytes_per_token\":28672,\"layers\":28,\"kv_heads\":2,\"attention_heads\":12,\"hidden_size\":1536,\"attention_selected\":\"sdpa\"},\"environment\":{\"torch\":\"2.14.0+cu130\",\"transformers\":\"4.51.3\",\"cuda\":\"13.0\",\"gpu\":\"NVIDIA GeForce RTX 5070 Ti\",\"dtype\":\"float16\",\"attention_requested\":\"sdpa\",\"logits_to_keep\":1},\"generation\":{\"max_new_tokens\":24,\"do_sample\":false,\"batch\":1,\"eos_token_ids\":[151645,151643]},\"measurement\":{\"warmup_requests\":1,\"measured_requests\":1,\"interval\":\"synchronized wall and CUDA events: forward + argmax; excludes CPU token extraction/text rendering/resource checks\",\"memory_peak_scope\":\"reset before each individual forward+argmax; model remains loaded\",\"limitations\":[\"single request, no distribution or performance guarantee\",\"per-token synchronization and inspection alter the execution\",\"not HTTP TTFT/TPOT, not summed CUDA kernel time, not protocol v2\"]},\"resource_samples\":[{\"stage\":\"before_load\",\"host_available_bytes\":74161643520,\"device_used_bytes\":1356398592,\"allocated_bytes\":0,\"reserved_bytes\":0,\"peak_allocated_bytes\":0,\"peak_reserved_bytes\":0},{\"stage\":\"post_load\",\"host_available_bytes\":71310524416,\"device_used_bytes\":4642635776,\"allocated_bytes\":3088347136,\"reserved_bytes\":3286237184,\"peak_allocated_bytes\":3088347136,\"peak_reserved_bytes\":3286237184},{\"stage\":\"after_warmup\",\"host_available_bytes\":71274737664,\"device_used_bytes\":4737007616,\"allocated_bytes\":3121901568,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3125901824,\"peak_reserved_bytes\":3323985920},{\"stage\":\"after_replay\",\"host_available_bytes\":71274737664,\"device_used_bytes\":4737007616,\"allocated_bytes\":3123364864,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920},{\"stage\":\"after_release\",\"host_available_bytes\":71274799104,\"device_used_bytes\":1446576128,\"allocated_bytes\":33554432,\"reserved_bytes\":33554432,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920}],\"prompt\":{\"messages\":[{\"role\":\"system\",\"content\":\"You are a helpful assistant. Reply with only the requested numbers.\"},{\"role\":\"user\",\"content\":\"Continue the sequence with three numbers: 2, 4, 6,\"}],\"rendered_chat\":\"<|im_start|>system\\nYou are a helpful assistant. Reply with only the requested numbers.<|im_end|>\\n<|im_start|>user\\nContinue the sequence with three numbers: 2, 4, 6,<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,2610,525,264,10950,17847,13,17841,448,1172,279,11223,5109,13,151645,198,151644,872,198,23526,279,8500,448,2326,5109,25,220,17,11,220,19,11,220,21,11,151645,198,151644,77091,198],\"input_token_count\":42,\"input_tokens\":[{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":0},{\"id\":8948,\"piece\":\"system\",\"text\":\"system\",\"index\":1},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":2},{\"id\":2610,\"piece\":\"You\",\"text\":\"You\",\"index\":3},{\"id\":525,\"piece\":\"Ġare\",\"text\":\" are\",\"index\":4},{\"id\":264,\"piece\":\"Ġa\",\"text\":\" a\",\"index\":5},{\"id\":10950,\"piece\":\"Ġhelpful\",\"text\":\" helpful\",\"index\":6},{\"id\":17847,\"piece\":\"Ġassistant\",\"text\":\" assistant\",\"index\":7},{\"id\":13,\"piece\":\".\",\"text\":\".\",\"index\":8},{\"id\":17841,\"piece\":\"ĠReply\",\"text\":\" Reply\",\"index\":9},{\"id\":448,\"piece\":\"Ġwith\",\"text\":\" with\",\"index\":10},{\"id\":1172,\"piece\":\"Ġonly\",\"text\":\" only\",\"index\":11},{\"id\":279,\"piece\":\"Ġthe\",\"text\":\" the\",\"index\":12},{\"id\":11223,\"piece\":\"Ġrequested\",\"text\":\" requested\",\"index\":13},{\"id\":5109,\"piece\":\"Ġnumbers\",\"text\":\" numbers\",\"index\":14},{\"id\":13,\"piece\":\".\",\"text\":\".\",\"index\":15},{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\",\"index\":16},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":17},{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":18},{\"id\":872,\"piece\":\"user\",\"text\":\"user\",\"index\":19},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":20},{\"id\":23526,\"piece\":\"Continue\",\"text\":\"Continue\",\"index\":21},{\"id\":279,\"piece\":\"Ġthe\",\"text\":\" the\",\"index\":22},{\"id\":8500,\"piece\":\"Ġsequence\",\"text\":\" sequence\",\"index\":23},{\"id\":448,\"piece\":\"Ġwith\",\"text\":\" with\",\"index\":24},{\"id\":2326,\"piece\":\"Ġthree\",\"text\":\" three\",\"index\":25},{\"id\":5109,\"piece\":\"Ġnumbers\",\"text\":\" numbers\",\"index\":26},{\"id\":25,\"piece\":\":\",\"text\":\":\",\"index\":27},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":28},{\"id\":17,\"piece\":\"2\",\"text\":\"2\",\"index\":29},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":30},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":31},{\"id\":19,\"piece\":\"4\",\"text\":\"4\",\"index\":32},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":33},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":34},{\"id\":21,\"piece\":\"6\",\"text\":\"6\",\"index\":35},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":36},{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\",\"index\":37},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":38},{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":39},{\"id\":77091,\"piece\":\"assistant\",\"text\":\"assistant\",\"index\":40},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":41}]},\"replay\":{\"output_token_ids\":[23,11,220,16,15,11,220,16,17,151645],\"output_text\":\"8, 10, 12\",\"output_text_with_special_tokens\":\"8, 10, 12<|im_end|>\",\"stop_reason\":\"eos\",\"steps\":[{\"output_index\":1,\"phase\":\"prefill\",\"input_shape\":[1,42],\"input_token_ids\":[151644,8948,198,2610,525,264,10950,17847,13,17841,448,1172,279,11223,5109,13,151645,198,151644,872,198,23526,279,8500,448,2326,5109,25,220,17,11,220,19,11,220,21,11,151645,198,151644,77091,198],\"kv_tokens_before\":0,\"kv_tokens_after\":42,\"selected_token\":{\"id\":23,\"piece\":\"8\",\"text\":\"8\"},\"decoded_prefix\":\"8\",\"wall_ms\":19.63335300001745,\"cuda_event_ms\":19.085664749145508,\"memory\":{\"allocated_bytes\":3123106304,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3125901824,\"peak_reserved_bytes\":3323985920}},{\"output_index\":2,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[23],\"kv_tokens_before\":42,\"kv_tokens_after\":43,\"selected_token\":{\"id\":11,\"piece\":\",\",\"text\":\",\"},\"decoded_prefix\":\"8,\",\"wall_ms\":81.22387200000958,\"cuda_event_ms\":81.15577697753906,\"memory\":{\"allocated_bytes\":3123135488,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123442176,\"peak_reserved_bytes\":3323985920}},{\"output_index\":3,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[11],\"kv_tokens_before\":43,\"kv_tokens_after\":44,\"selected_token\":{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \"},\"decoded_prefix\":\"8, \",\"wall_ms\":28.601944000001822,\"cuda_event_ms\":28.543039321899414,\"memory\":{\"allocated_bytes\":3123164160,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123470848,\"peak_reserved_bytes\":3323985920}},{\"output_index\":4,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[220],\"kv_tokens_before\":44,\"kv_tokens_after\":45,\"selected_token\":{\"id\":16,\"piece\":\"1\",\"text\":\"1\"},\"decoded_prefix\":\"8, 1\",\"wall_ms\":20.453992999819093,\"cuda_event_ms\":20.372831344604492,\"memory\":{\"allocated_bytes\":3123192832,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123499520,\"peak_reserved_bytes\":3323985920}},{\"output_index\":5,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[16],\"kv_tokens_before\":45,\"kv_tokens_after\":46,\"selected_token\":{\"id\":15,\"piece\":\"0\",\"text\":\"0\"},\"decoded_prefix\":\"8, 10\",\"wall_ms\":33.23044800004027,\"cuda_event_ms\":33.17951965332031,\"memory\":{\"allocated_bytes\":3123221504,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123528192,\"peak_reserved_bytes\":3323985920}},{\"output_index\":6,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[15],\"kv_tokens_before\":46,\"kv_tokens_after\":47,\"selected_token\":{\"id\":11,\"piece\":\",\",\"text\":\",\"},\"decoded_prefix\":\"8, 10,\",\"wall_ms\":16.885007999917434,\"cuda_event_ms\":16.802719116210938,\"memory\":{\"allocated_bytes\":3123250176,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123556864,\"peak_reserved_bytes\":3323985920}},{\"output_index\":7,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[11],\"kv_tokens_before\":47,\"kv_tokens_after\":48,\"selected_token\":{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \"},\"decoded_prefix\":\"8, 10, \",\"wall_ms\":68.2125509999878,\"cuda_event_ms\":68.14598083496094,\"memory\":{\"allocated_bytes\":3123278848,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123591680,\"peak_reserved_bytes\":3323985920}},{\"output_index\":8,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[220],\"kv_tokens_before\":48,\"kv_tokens_after\":49,\"selected_token\":{\"id\":16,\"piece\":\"1\",\"text\":\"1\"},\"decoded_prefix\":\"8, 10, 1\",\"wall_ms\":17.700886000056926,\"cuda_event_ms\":17.647199630737305,\"memory\":{\"allocated_bytes\":3123307520,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123626496,\"peak_reserved_bytes\":3323985920}},{\"output_index\":9,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[16],\"kv_tokens_before\":49,\"kv_tokens_after\":50,\"selected_token\":{\"id\":17,\"piece\":\"2\",\"text\":\"2\"},\"decoded_prefix\":\"8, 10, 12\",\"wall_ms\":55.26893700016444,\"cuda_event_ms\":55.212703704833984,\"memory\":{\"allocated_bytes\":3123336192,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123661312,\"peak_reserved_bytes\":3323985920}},{\"output_index\":10,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[17],\"kv_tokens_before\":50,\"kv_tokens_after\":51,\"selected_token\":{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\"},\"decoded_prefix\":\"8, 10, 12<|im_end|>\",\"wall_ms\":16.745422999974835,\"cuda_event_ms\":16.682655334472656,\"memory\":{\"allocated_bytes\":3123364864,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920}}],\"final_kv_tokens\":51,\"instrumented_request_wall_ms\":369.3175700000211}}")
prompt = RECORDED_PROMPT['prompt']
print('RECORDED INPUT — no inference in this cell')
for message in prompt['messages']:
    print(message['role'].upper() + ':', message['content'])
print('\nEXACT RENDERED CHAT (repr exposes newlines):')
print(repr(prompt['rendered_chat']))
print('\nGPU input shape:', [1, prompt['input_token_count']])
print('position | token ID | tokenizer piece | decoded single-ID text')
for token in prompt['input_tokens']:
    print(token['index'], token['id'], repr(token['piece']), repr(token['text']), sep=' | ')
assert len(prompt['input_token_ids']) == prompt['input_token_count']
assert [token['id'] for token in prompt['input_tokens']] == prompt['input_token_ids']


### Reveal one forward at a time

Set REVEAL_OUTPUTS to 1, run the cell, predict the next input ID/shape and retained length, then increase the number. No model executes: you are revealing already recorded evidence. Token IDs are selected output values; choosing an ID does not create its K/V until the next forward consumes it. Special stop tokens can count as selected IDs without adding visible text.

The wall and CUDA-event numbers below are **individual instrumented observations**, not medians, a stable target, kernel-time sums or client streaming timings. Model loading, tokenization, decoding/printing the row and HTTP are not included.


In [ ]:
REVEAL_OUTPUTS = 1  # change to len(RECORDED_PROMPT['replay']['steps']) for the complete replay
replay = RECORDED_PROMPT['replay']
assert 1 <= REVEAL_OUTPUTS <= len(replay['steps'])
for step in replay['steps'][:REVEAL_OUTPUTS]:
    token = step['selected_token']
    print('\nOutput choice', step['output_index'], '| phase:', step['phase'])
    print('forward input shape:', step['input_shape'], '| input IDs:', step['input_token_ids'])
    print('selected:', token['id'], repr(token['text']))
    print('KV positions:', step['kv_tokens_before'], '->', step['kv_tokens_after'])
    print('cumulative text:', repr(step['decoded_prefix']))
    print('wall ms:', round(step['wall_ms'], 3), '| CUDA-event interval ms:', round(step['cuda_event_ms'], 3))
    assert step['kv_tokens_after'] == prompt['input_token_count'] + step['output_index'] - 1
if REVEAL_OUTPUTS == len(replay['steps']):
    print('\nFinal visible output:', repr(replay['output_text']))
    print('Selected IDs:', replay['output_token_ids'], '| stop reason:', replay['stop_reason'])
    assert replay['final_kv_tokens'] == prompt['input_token_count'] + len(replay['output_token_ids']) - 1
else:
    print('\nPredict the next row, then increment REVEAL_OUTPUTS. No GPU is used.')


### Before the GPU route: three measured comparisons to reconstruct

These are canonical **protocol-v2** measurements, not timings for the sentence above. Source: [30 September raw trials and protocol](https://levshaazz.github.io/llm-serving-mastery/seminars/runs/2026-09-30-topic02-protocol-v2/README.md). RTX 5070 Ti, FP16, five trials per case; brackets are observed min–max, not p95 or confidence intervals.

| Pinned 1.5B case | Measured observation | Predict / explain |
|---|---|---|
| B1/P4096/O32 | Prefill 163.7 ms [163.4–163.8]; 31-step decode 1039.1 ms [763.9–1154.4] | Why can 4096 known positions finish before 31 dependent steps? |
| B1→B4, P128/O32 | Aggregate 36.9 [24.7–37.1] → 155.2 [103.3–162.5] tok/s; per-lane cadence 27.1 [26.9–40.5] → 25.8 [24.6–38.7] ms | Reconstruct numerators 31 and 124. Did one lane become 4.2× faster? |
| B1/P4096, O32→O128 | Calculated final KV 112.85 → 115.47 MiB; measured allocation peak 3349.34 MiB in both | Why can final KV grow without a higher whole-phase peak? |

The 1.5B coefficient is 28 KiB/token (28 layers, 2 KV heads, head dimension 128, FP16), **not** the 3B transfer exercise's 36 KiB/token. Final retained length is P+O−1. No row establishes serving capacity or an HTTP SLO.

**Unresolved control:** identical P128/B1 prefills under O32 and O128 labels have medians 22.3 and 64.6 ms, although the future output count is not part of the prefill forward. Preserve this discrepancy rather than inventing its cause.

Only continue below when ready for your own bounded GPU experiment. Recorded replay may be your classroom fallback, never a replacement for required GPU evidence.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess, sys
pinned = {'transformers': '4.51.3', 'accelerate': '1.6.0', 'huggingface_hub': '0.30.2', 'safetensors': '0.5.3'}
for name, wanted in pinned.items():
    try: installed = version(name) == wanted
    except PackageNotFoundError: installed = False
    if not installed:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'{name}=={wanted}'])
# If you previously imported a different Transformers version, restart the kernel now.


## 1. Read the measurement boundary

The following cell is the complete editable implementation. Prefill includes one forward and selection of output y1; decode includes O−1 forwards and greedy selections. The last selected token has no KV until it is fed back, so retained length is P+O−1. EOS is ignored to hold lengths fixed. Inputs are repeated synthetic token IDs, with no padding or chat-template claim.

Each phase records synchronized wall time and a CUDA-event interval. **An event interval is not a sum of kernel durations:** host launch gaps can lie between events. Neither timer includes tokenization, model loading, HTTP, queueing or delivery. Do not call these values server TTFT/TPOT or p95.

Last-position logits avoid computing unused vocabulary projections for every prompt position. Full-shape warmup is outside timing; cases are shuffled within five rounds, one model at a time. Allocator pools stay warm across trials. Post-trial live-allocation recovery and host RAM are checked; peaks are recorded. Gates are stop rules, not an OS memory sandbox. Instructor runs also use an external memory limit and wall timeout.


In [ ]:
"""Topic 02 protocol v2. Embedded verbatim in the standalone public notebook.

No downloads on import; no CUDA required for the arithmetic/contract functions.
This is a guarded Python phase experiment, not an HTTP service benchmark.
"""
import gc
import json
import math
import os
import platform
import random
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

PROTOCOL = "topic02-phase-v2"
MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]
TRIALS = 5
ORDER_SEED = 20260930
GIB = 1024 ** 3


def kv_bytes_per_token(config, element_bytes=2):
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    return 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * element_bytes


def attention_pairs(prompt, output):
    """Decode only: output 1 is selected from prefill; each later query includes itself."""
    steps = output - 1
    return steps * prompt + steps * (steps + 1) // 2


def trial_order(cases=CASES, trials=TRIALS):
    rng = random.Random(ORDER_SEED)
    schedule = []
    for repeat in range(trials):
        order = list(cases)
        rng.shuffle(order)
        schedule.extend((repeat, *case) for case in order)
    return schedule


def checkpoint(path, data):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def host_available_bytes():
    # Colab and the instructor's WSL environment are Linux. Fail closed on unknown hosts.
    meminfo = Path("/proc/meminfo")
    if not meminfo.exists():
        raise RuntimeError("Run this CUDA lab on Linux/Colab/WSL; host memory gate is unavailable")
    for line in meminfo.read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("MemAvailable is unavailable")


def process_rss_bytes():
    return int(Path("/proc/self/statm").read_text().split()[1]) * os.sysconf("SC_PAGE_SIZE")


def resource_gate(torch, baseline_allocated=None, baseline_rss=None):
    allocated = torch.cuda.memory_allocated()
    free, total = torch.cuda.mem_get_info()
    limit = min(12 * GIB, int(total * 0.80))
    if allocated > limit or total - free > min(12 * GIB, int(total * 0.90)):
        raise RuntimeError("STOP: GPU memory budget exceeded; retain partial evidence and inspect other processes")
    if host_available_bytes() < 2 * GIB:
        raise RuntimeError("STOP: less than 2 GiB host RAM available")
    if baseline_allocated is not None and allocated > baseline_allocated + 128 * 1024 ** 2:
        raise RuntimeError("STOP: live CUDA allocations failed to return to the post-load baseline")
    if baseline_rss is not None and process_rss_bytes() > baseline_rss + 2 * GIB:
        raise RuntimeError("STOP: process RSS grew by more than 2 GiB")
    return {"allocated_bytes": allocated, "reserved_bytes": torch.cuda.memory_reserved(),
            "device_used_bytes": total - free, "host_available_bytes": host_available_bytes(),
            "process_rss_bytes": process_rss_bytes()}


def exact_input(torch, tokenizer, batch, length):
    tokens = tokenizer("Explain GPU memory reuse and token dependencies. ", add_special_tokens=False)["input_ids"]
    row = (tokens * math.ceil(length / len(tokens)))[:length]
    return torch.tensor([row] * batch, dtype=torch.long, device="cuda")


def phase_trial(torch, model, ids, output_tokens):
    """No HTTP, tokenization, model load or profiler inside these intervals.

    Prefill includes selecting y1. Decode includes O-1 forwards and greedy selections.
    CUDA-event elapsed time spans host launch gaps too; it is not summed kernel time.
    Exact-length synthetic generation deliberately ignores EOS; it is not a quality test.
    """
    out = past = token = None
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    try:
        with torch.inference_mode():
            wall_start = time.perf_counter()
            start.record()
            out = model(ids, use_cache=True, logits_to_keep=1)
            token = out.logits[:, -1].argmax(-1, keepdim=True)
            past = out.past_key_values
            out = None  # Do not retain a full prompt-logit tensor during decode.
            end.record()
            end.synchronize()
            prefill_wall_ms = (time.perf_counter() - wall_start) * 1000
            prefill_event_ms = start.elapsed_time(end)
            prefill_peak = torch.cuda.max_memory_allocated()
            wall_start = time.perf_counter()
            start.record()
            for _ in range(output_tokens - 1):
                out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                token = out.logits[:, -1].argmax(-1, keepdim=True)
                past = out.past_key_values
                out = None
            end.record()
            end.synchronize()
            decode_wall_ms = (time.perf_counter() - wall_start) * 1000
            decode_event_ms = start.elapsed_time(end)
            retained = int(past.get_seq_length())
            expected = ids.shape[1] + output_tokens - 1
            if retained != expected:
                raise AssertionError(f"KV sequence length {retained} != {expected}")
            steps = output_tokens - 1
            return {
                "prefill_wall_ms": prefill_wall_ms, "prefill_event_ms": prefill_event_ms,
                "decode_wall_ms": decode_wall_ms, "decode_event_ms": decode_event_ms,
                "decode_steps_per_sequence": steps, "generated_tokens": ids.shape[0] * output_tokens,
                "retained_tokens_per_sequence": retained,
                "decode_aggregate_tokens_per_s": ids.shape[0] * steps * 1000 / decode_wall_ms,
                "decode_mean_step_ms": decode_wall_ms / steps,
                "prefill_peak_allocated_bytes": prefill_peak,
                "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
            }
    finally:
        out = past = token = None


def run_matrix(model_paths, output_dir, *, smoke=False):
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required; CPU checks cannot replace this run")
    if transformers.__version__ != "4.51.3":
        raise RuntimeError("Use the pinned Transformers 4.51.3 runtime")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)  # Never silently overwrite a previous attempt.
    output_path = output_dir / "topic02-benchmark.json"
    cases = [CASES[0]] if smoke else CASES
    model_ids = list(MODELS)[:1] if smoke else list(MODELS)
    data = {"protocol": PROTOCOL, "status": "running", "scope": "smoke" if smoke else "core",
            "started_at": datetime.now(timezone.utc).isoformat(),
            "environment": {"python": platform.python_version(), "torch": torch.__version__,
                            "transformers": transformers.__version__, "cuda": torch.version.cuda,
                            "gpu": torch.cuda.get_device_name(), "dtype": "float16",
                            "attention_requested": "sdpa", "logits_to_keep": 1},
            "models": MODELS, "order_seed": ORDER_SEED, "trials": 1 if smoke else TRIALS,
            "cases": [list(c) for c in cases], "results": [], "resource_samples": [],
            "measurement": "synchronized wall + CUDA-event intervals, greedy forced length, no HTTP",
            "model_details": {}}
    checkpoint(output_path, data)
    model = tokenizer = ids = None
    try:
        data["resource_samples"].append(resource_gate(torch))
        for model_id in model_ids:
            try:
                model = AutoModelForCausalLM.from_pretrained(
                    model_paths[model_id], local_files_only=True, torch_dtype=torch.float16,
                    attn_implementation="sdpa").to("cuda").eval()
                tokenizer = AutoTokenizer.from_pretrained(model_paths[model_id], local_files_only=True)
                torch.cuda.synchronize()
                baseline = torch.cuda.memory_allocated()
                rss = process_rss_bytes()
                data["model_details"][model_id] = {
                    "kv_bytes_per_token": kv_bytes_per_token(model.config),
                    "post_load_allocated_bytes": baseline,
                    "attention_selected": model.config._attn_implementation,
                    "vocab_size": model.config.vocab_size,
                }
                resource_gate(torch, baseline, rss)
                # Warm each actual shape, including the complete decode path. Warmup is not a trial.
                for batch, prompt, output in cases:
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    resource_gate(torch, baseline, rss)
                for repeat, batch, prompt, output in trial_order(cases, data["trials"]):
                    resource_gate(torch, baseline, rss)
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    result = phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    result.update(model=model_id, batch=batch, input_tokens=prompt,
                                  output_tokens=output, repeat=repeat)
                    data["results"].append(result)
                    checkpoint(output_path, data)  # Preserve the trial before checking recovery.
                    data["resource_samples"].append(resource_gate(torch, baseline, rss))
                    if result["peak_allocated_bytes"] > min(12 * GIB, torch.cuda.get_device_properties(0).total_memory * .80):
                        raise RuntimeError("STOP: an in-process allocation peak exceeded the budget")
                    checkpoint(output_path, data)
                    print(model_id, (batch, prompt, output), "trial", repeat + 1, flush=True)
            finally:
                model = tokenizer = ids = None
                gc.collect()
                torch.cuda.empty_cache()  # Between models only, never between timed trials.
                torch.cuda.synchronize()
                data["resource_samples"].append(resource_gate(torch))
        data["status"] = "complete"
    except Exception as exc:
        data["status"] = "failed"
        data["error_type"] = type(exc).__name__
        raise
    finally:
        data["finished_at"] = datetime.now(timezone.utc).isoformat()
        checkpoint(output_path, data)
    return data


def summarize(data):
    rows = []
    groups = {}
    for trial in data["results"]:
        key = tuple(trial[k] for k in ("model", "batch", "input_tokens", "output_tokens"))
        groups.setdefault(key, []).append(trial)
    for key, trials in groups.items():
        row = dict(zip(("model", "batch", "input_tokens", "output_tokens"), key))
        row["n"] = len(trials)
        for metric in ("prefill_wall_ms", "decode_wall_ms", "decode_mean_step_ms", "decode_aggregate_tokens_per_s"):
            values = [t[metric] for t in trials]
            row[metric] = {"median": statistics.median(values), "min": min(values), "max": max(values)}
        rows.append(row)
    return rows


def focused_profiles(model_path, output_dir):
    """Two independent traces: P=128 and P=4096, each prefill + seven decode steps.

    No profile_memory/record_shapes by default: these add overhead/retain references.
    The trace proves event ordering/operator attribution, not unprofiled latency or HBM bandwidth.
    """
    import torch
    from torch.profiler import profile, record_function, ProfilerActivity
    from transformers import AutoModelForCausalLM, AutoTokenizer
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    model = tokenizer = ids = out = past = token = None
    summaries = []
    try:
        resource_gate(torch)
        model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True,
                    torch_dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        baseline = torch.cuda.memory_allocated()
        for prompt in (128, 4096):
            ids = exact_input(torch, tokenizer, 1, prompt)
            phase_trial(torch, model, ids, 8)
            with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                with record_function("topic02_prefill"):
                    out = model(ids, use_cache=True, logits_to_keep=1)
                    past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                    out = None
                with record_function("topic02_decode_7_steps"):
                    for _ in range(7):
                        out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                        past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                        out = None
                torch.cuda.synchronize()
            prof.export_chrome_trace(str(output_dir / f"topic02-trace-p{prompt}.json"))
            events = sorted(prof.key_averages(), key=lambda e: e.self_device_time_total, reverse=True)[:12]
            summaries.append({"input_tokens": prompt, "output_tokens": 8, "model": list(MODELS)[0],
                              "operators": [{"name": e.key, "calls": e.count,
                                             "self_device_us": e.self_device_time_total} for e in events]})
            ids = out = past = token = prof = events = None
            gc.collect()
            resource_gate(torch, baseline)
            checkpoint(output_dir / "topic02-profile-summary.json", summaries)
    finally:
        model = tokenizer = ids = out = past = token = None
        gc.collect()
        torch.cuda.empty_cache()
    return summaries


## 2. Resolve pinned models once

Set **LSM_MODEL_CACHE** to persistent storage before running this cell (for example a mounted persistent volume). The default cache is persistent only while this runtime survives. An offline instructor run sets HF_HUB_OFFLINE=1; missing snapshots then fail without downloading. Never delete model caches when making a fresh evidence directory.


In [ ]:
from huggingface_hub import snapshot_download
cache = Path(os.environ.get('LSM_MODEL_CACHE', str(Path.home() / '.cache' / 'lsm-models')))
cache.mkdir(parents=True, exist_ok=True)
MODEL_PATHS = {}
for model_id, revision in MODELS.items():
    try:
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache, local_files_only=True)
    except FileNotFoundError:
        if os.environ.get('HF_HUB_OFFLINE') == '1':
            raise RuntimeError('Pinned model is missing from the offline cache')
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache)
    MODEL_PATHS[model_id] = resolved
print('Pinned snapshots resolved; local paths remain private.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EVIDENCE = Path('evidence') / RUN_ID


### Optional: execute that exact prompt locally, once

**Default: off. Not a required artifact or a substitute for the 50-trial matrix.** The recorded replay above always works without this cell. The embedded helper is the same canonical source used for the instructor demonstration and consumes the already resolved immutable 1.5B snapshot; it never downloads a model. It is deliberately limited to the instructor-style Linux/WSL safety envelope: at least **24 GiB available host RAM**, at most 12 GiB device use, one GPU process, one warmup request and one measured request, at most 24 generated IDs. Many Colab machines do not satisfy the host gate; do not lower it to force this optional demonstration.

If enabled, keep the notebook's existing safety guidance and your host's external process memory/time limits. The helper releases the model on exit and rechecks host/device headroom; this is not an assertion that allocation returned to its initial baseline. Stop on any failure instead of blindly retrying. Per-step synchronization, token extraction and text inspection alter the execution; this is an explanation of state, not an additional speed benchmark. Exact timing is not expected to reproduce the instructor's values, and implementation/runtime changes can change greedy output.

All required matrix cells below remain unchanged and use their own protocol, outputs and validator. Do not run live demo and matrix in parallel.


In [ ]:
RUN_CONCRETE_PROMPT_LIVE = False
if RUN_CONCRETE_PROMPT_LIVE:
    optional_dir = EVIDENCE / 'optional-concrete-prompt'
    optional_dir.mkdir(parents=True, exist_ok=False)  # reject an existing attempt before GPU work
    # Isolated namespace prevents the demo's constants from changing the required lab protocol.
    demo_namespace = {name: globals()[name] for name in ('GIB', 'MODELS', 'host_available_bytes', 'kv_bytes_per_token')}
    exec("\"\"\"One real request, recorded token by token. Not a benchmark or protocol-v2 trial.\n\nImport is CPU-only. Live execution requires Linux/WSL, an idle CUDA GPU, a pinned\nlocal snapshot and >=24 GiB available host RAM. No downloads are performed here.\n\"\"\"\nimport gc\nimport math\nimport time\nfrom pathlib import Path\n\n\nPROTOCOL = \"topic02-concrete-prompt-v1\"\nMODEL_ID = \"Qwen/Qwen2.5-1.5B-Instruct\"\nMODEL_REVISION = MODELS[MODEL_ID]\nMESSAGES = [\n    {\"role\": \"system\", \"content\": \"You are a helpful assistant. Reply with only the requested numbers.\"},\n    {\"role\": \"user\", \"content\": \"Continue the sequence with three numbers: 2, 4, 6,\"},\n]\nMAX_NEW_TOKENS = 24\n\n\ndef token_record(tokenizer, token_id, index=None):\n    value = {\"id\": int(token_id), \"piece\": tokenizer.convert_ids_to_tokens(int(token_id)),\n             \"text\": tokenizer.decode([int(token_id)], skip_special_tokens=False,\n                                      clean_up_tokenization_spaces=False)}\n    if index is not None:\n        value[\"index\"] = index\n    return value\n\n\ndef validate_replay(data):\n    \"\"\"CPU-only contract: visible selections lag the retained KV cache by one.\"\"\"\n    def positive_duration(value):\n        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:\n            raise ValueError(\"measured duration must be finite and positive\")\n\n    def byte_counter(value):\n        # JSON byte counts are integers. Reject bool as well as float NaN/Inf;\n        # relying on ordinary comparisons alone lets NaN peaks pass unnoticed.\n        if type(value) is not int or value < 0:\n            raise ValueError(\"memory counter must be a finite nonnegative integer\")\n\n    if data[\"protocol\"] != PROTOCOL:\n        raise ValueError(\"unexpected replay protocol\")\n    prompt = data[\"prompt\"]\n    p = prompt[\"input_token_count\"]\n    if p != len(prompt[\"input_token_ids\"]) or p < 1:\n        raise ValueError(\"prompt length mismatch\")\n    if [t[\"id\"] for t in prompt[\"input_tokens\"]] != prompt[\"input_token_ids\"]:\n        raise ValueError(\"prompt pieces do not match IDs\")\n    replay = data[\"replay\"]\n    positive_duration(replay[\"instrumented_request_wall_ms\"])\n    steps = replay[\"steps\"]\n    if not 1 <= len(steps) <= data[\"generation\"][\"max_new_tokens\"] <= MAX_NEW_TOKENS:\n        raise ValueError(\"output length exceeds the bounded contract\")\n    if [s[\"selected_token\"][\"id\"] for s in steps] != replay[\"output_token_ids\"]:\n        raise ValueError(\"selected IDs mismatch\")\n    previous_id = None\n    for index, step in enumerate(steps, 1):\n        expected_input = prompt[\"input_token_ids\"] if index == 1 else [previous_id]\n        if step[\"output_index\"] != index or step[\"input_token_ids\"] != expected_input:\n            raise ValueError(\"generation dependency mismatch\")\n        if step[\"input_shape\"] != [1, len(expected_input)]:\n            raise ValueError(\"input shape mismatch\")\n        if step[\"phase\"] != (\"prefill\" if index == 1 else \"decode\"):\n            raise ValueError(\"phase mismatch\")\n        if step[\"kv_tokens_before\"] != (0 if index == 1 else p + index - 2):\n            raise ValueError(\"cache before mismatch\")\n        if step[\"kv_tokens_after\"] != p + index - 1:\n            raise ValueError(\"cache after mismatch\")\n        for key in (\"wall_ms\", \"cuda_event_ms\"):\n            positive_duration(step[key])\n        mem = step[\"memory\"]\n        for key in (\"allocated_bytes\", \"reserved_bytes\", \"peak_allocated_bytes\", \"peak_reserved_bytes\"):\n            byte_counter(mem[key])\n        if not (0 <= mem[\"allocated_bytes\"] <= mem[\"reserved_bytes\"]):\n            raise ValueError(\"allocator nesting mismatch\")\n        if mem[\"peak_allocated_bytes\"] < mem[\"allocated_bytes\"] or mem[\"peak_reserved_bytes\"] < mem[\"reserved_bytes\"]:\n            raise ValueError(\"peak memory is below current memory\")\n        previous_id = step[\"selected_token\"][\"id\"]\n    if replay[\"final_kv_tokens\"] != p + len(steps) - 1:\n        raise ValueError(\"final KV lag mismatch\")\n    eos = data[\"generation\"][\"eos_token_ids\"]\n    if any(t in eos for t in replay[\"output_token_ids\"][:-1]):\n        raise ValueError(\"generation continued after EOS\")\n    if replay[\"stop_reason\"] == \"eos\":\n        if replay[\"output_token_ids\"][-1] not in eos:\n            raise ValueError(\"EOS stop without EOS\")\n    elif replay[\"stop_reason\"] != \"max_new_tokens\" or len(steps) != data[\"generation\"][\"max_new_tokens\"]:\n        raise ValueError(\"invalid stop reason\")\n    for sample in data[\"resource_samples\"]:\n        for key in (\"host_available_bytes\", \"device_used_bytes\", \"allocated_bytes\", \"reserved_bytes\",\n                    \"peak_allocated_bytes\", \"peak_reserved_bytes\"):\n            byte_counter(sample[key])\n    return True\n\n\ndef memory_sample(torch):\n    return {\"allocated_bytes\": torch.cuda.memory_allocated(),\n            \"reserved_bytes\": torch.cuda.memory_reserved(),\n            \"peak_allocated_bytes\": torch.cuda.max_memory_allocated(),\n            \"peak_reserved_bytes\": torch.cuda.max_memory_reserved()}\n\n\ndef resource_gate(torch):\n    available = host_available_bytes()\n    free, total = torch.cuda.mem_get_info()\n    used = total - free\n    if available < 24 * GIB or used > 12 * GIB or torch.cuda.max_memory_allocated() > 12 * GIB:\n        raise RuntimeError(\"STOP: require >=24 GiB available RAM and <=12 GiB device-used memory\")\n    return {\"host_available_bytes\": available, \"device_used_bytes\": used, **memory_sample(torch)}\n\n\ndef run_demo(model_path):\n    \"\"\"Warm one request, then record one instrumented replay; local snapshot only.\n\n    Each interval encloses model forward + argmax. Synchronization at every token,\n    CPU ID extraction, token decoding and resource checks perturb the request;\n    they make this explanation inspectable, not a valid service throughput test.\n    \"\"\"\n    import torch\n    import transformers\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    model_path = Path(model_path)\n    if model_path.name != MODEL_REVISION:\n        raise RuntimeError(\"Only the pinned model snapshot is supported\")\n    if transformers.__version__ != \"4.51.3\" or not torch.cuda.is_available():\n        raise RuntimeError(\"Require Transformers 4.51.3 and a CUDA GPU\")\n    data = {\"protocol\": PROTOCOL, \"scope\": \"one instrumented natural-EOS request; not a benchmark\",\n            \"model\": {\"id\": MODEL_ID, \"revision\": MODEL_REVISION},\n            \"environment\": {\"torch\": torch.__version__, \"transformers\": transformers.__version__,\n                            \"cuda\": torch.version.cuda, \"gpu\": torch.cuda.get_device_name(),\n                            \"dtype\": \"float16\", \"attention_requested\": \"sdpa\", \"logits_to_keep\": 1},\n            \"generation\": {\"max_new_tokens\": MAX_NEW_TOKENS, \"do_sample\": False, \"batch\": 1},\n            \"measurement\": {\"warmup_requests\": 1, \"measured_requests\": 1,\n                \"interval\": \"synchronized wall and CUDA events: forward + argmax; excludes CPU token extraction/text rendering/resource checks\",\n                \"memory_peak_scope\": \"reset before each individual forward+argmax; model remains loaded\",\n                \"limitations\": [\"single request, no distribution or performance guarantee\",\n                    \"per-token synchronization and inspection alter the execution\",\n                    \"not HTTP TTFT/TPOT, not summed CUDA kernel time, not protocol v2\"]},\n            \"resource_samples\": []}\n    model = tokenizer = ids = past = out = token = current = None\n    try:\n        data[\"resource_samples\"].append({\"stage\": \"before_load\", **resource_gate(torch)})\n        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)\n        model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True,\n            torch_dtype=torch.float16, attn_implementation=\"sdpa\").to(\"cuda\").eval()\n        chat = tokenizer.apply_chat_template(MESSAGES, tokenize=False, add_generation_prompt=True)\n        input_ids = tokenizer(chat, add_special_tokens=False)[\"input_ids\"]\n        ids = torch.tensor([input_ids], dtype=torch.long, device=\"cuda\")\n        p = len(input_ids)\n        eos = model.generation_config.eos_token_id\n        eos = [eos] if isinstance(eos, int) else list(eos)\n        data[\"generation\"][\"eos_token_ids\"] = eos\n        data[\"prompt\"] = {\"messages\": MESSAGES, \"rendered_chat\": chat,\n            \"input_token_ids\": input_ids, \"input_token_count\": p,\n            \"input_tokens\": [token_record(tokenizer, t, i) for i, t in enumerate(input_ids)]}\n        data[\"model\"].update(kv_bytes_per_token=kv_bytes_per_token(model.config),\n            layers=model.config.num_hidden_layers, kv_heads=model.config.num_key_value_heads,\n            attention_heads=model.config.num_attention_heads, hidden_size=model.config.hidden_size,\n            attention_selected=model.config._attn_implementation)\n        data[\"resource_samples\"].append({\"stage\": \"post_load\", **resource_gate(torch)})\n        # Same prompt and natural-EOS path, no timing/profiling retained as evidence.\n        with torch.inference_mode():\n            for index in range(MAX_NEW_TOKENS):\n                out = model(ids if index == 0 else token, past_key_values=past,\n                            use_cache=True, logits_to_keep=1)\n                token = out.logits[:, -1].argmax(-1, keepdim=True)\n                past = out.past_key_values\n                out = None\n                selected = int(token.item())\n                resource_gate(torch)\n                if selected in eos:\n                    break\n        token = past = None\n        gc.collect()\n        torch.cuda.synchronize()\n        # Keep the warmed allocator pool: never empty_cache between warmup and replay.\n        data[\"resource_samples\"].append({\"stage\": \"after_warmup\", **resource_gate(torch)})\n        generated, steps = [], []\n        replay_start = time.perf_counter()\n        with torch.inference_mode():\n            for index in range(MAX_NEW_TOKENS):\n                resource_gate(torch)\n                current = ids if index == 0 else token\n                before = 0 if past is None else int(past.get_seq_length())\n                torch.cuda.synchronize()\n                torch.cuda.reset_peak_memory_stats()\n                start = torch.cuda.Event(enable_timing=True)\n                end = torch.cuda.Event(enable_timing=True)\n                wall_start = time.perf_counter()\n                start.record()\n                out = model(current, past_key_values=past, use_cache=True, logits_to_keep=1)\n                token = out.logits[:, -1].argmax(-1, keepdim=True)\n                past = out.past_key_values\n                out = None\n                end.record()\n                end.synchronize()\n                wall_ms = (time.perf_counter() - wall_start) * 1000\n                memory = memory_sample(torch)\n                selected = int(token.item())\n                generated.append(selected)\n                steps.append({\"output_index\": index + 1, \"phase\": \"prefill\" if index == 0 else \"decode\",\n                    \"input_shape\": list(current.shape),\n                    \"input_token_ids\": input_ids if index == 0 else [generated[-2]],\n                    \"kv_tokens_before\": before, \"kv_tokens_after\": int(past.get_seq_length()),\n                    \"selected_token\": token_record(tokenizer, selected),\n                    \"decoded_prefix\": tokenizer.decode(generated, skip_special_tokens=False,\n                                                        clean_up_tokenization_spaces=False),\n                    \"wall_ms\": wall_ms, \"cuda_event_ms\": start.elapsed_time(end), \"memory\": memory})\n                resource_gate(torch)\n                if selected in eos:\n                    break\n        data[\"replay\"] = {\"output_token_ids\": generated,\n            \"output_text\": tokenizer.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False),\n            \"output_text_with_special_tokens\": tokenizer.decode(generated, skip_special_tokens=False,\n                                                                clean_up_tokenization_spaces=False),\n            \"stop_reason\": \"eos\" if generated[-1] in eos else \"max_new_tokens\",\n            \"steps\": steps, \"final_kv_tokens\": int(past.get_seq_length()),\n            \"instrumented_request_wall_ms\": (time.perf_counter() - replay_start) * 1000}\n        validate_replay(data)\n        data[\"resource_samples\"].append({\"stage\": \"after_replay\", **resource_gate(torch)})\n    finally:\n        model = tokenizer = ids = past = out = token = current = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        torch.cuda.synchronize()\n        data[\"resource_samples\"].append({\"stage\": \"after_release\", **resource_gate(torch)})\n    return data\n", demo_namespace)
    LIVE_PROMPT = demo_namespace['run_demo'](MODEL_PATHS['Qwen/Qwen2.5-1.5B-Instruct'])
    checkpoint(optional_dir / 'replay.json', LIVE_PROMPT)
    print('Actual messages:', LIVE_PROMPT['prompt']['messages'])
    print('Actual output:', repr(LIVE_PROMPT['replay']['output_text']))
    print('Stop reason:', LIVE_PROMPT['replay']['stop_reason'])
else:
    print('Optional live demonstration skipped; recorded replay remains available without a GPU.')


## 3. Startup + smoke — stop here if it fails

One small-model 128/32 case, one measured trial after warmup. Inspect memory recovery and the timings before continuing. A successful smoke is **not** the completed lab. If interrupted, retain its JSON and restart only after understanding the failure.


In [ ]:
SMOKE = run_matrix(MODEL_PATHS, EVIDENCE / 'smoke', smoke=True)
print(json.dumps(summarize(SMOKE), indent=2))
assert SMOKE['status'] == 'complete'


## 4. Required matrix + matched batch comparison

Expect 50 raw trials, checkpointed after each trial. A new attempt needs a new RUN_ID; existing results cannot be overwritten. Do not change shape, dtype, attention or counts mid-run. The two models load sequentially and are released even on an exception. The matrix is deliberately small; it does not establish production capacity.


In [ ]:
assert SMOKE['status'] == 'complete', 'Pass the smoke first'
RUN = run_matrix(MODEL_PATHS, EVIDENCE / 'core')
SUMMARY = summarize(RUN)
checkpoint(EVIDENCE / 'topic02-summary.json', SUMMARY)
for row in SUMMARY:
    print(json.dumps(row))


## 5. CPU artifact contract

This checker rejects incomplete matrices, duplicated trials, wrong revisions, nonfinite timings and off-by-one/rate errors. Passing validates structure and arithmetic, **not authenticity or the correctness of your bottleneck conclusion**. It is also available in the public template as scripts/validate_topic02.py.


In [ ]:
#!/usr/bin/env python3
"""CPU-only structure/arithmetic check, not a proof that a benchmark is genuine."""
import argparse
import json
import math
from pathlib import Path

MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]


def validate(data):
    if (data.get("protocol"), data.get("status"), data.get("scope"), data.get("trials")) != (
            "topic02-phase-v2", "complete", "core", 5):
        raise ValueError("Expected a completed core protocol-v2 run with five trials; smoke/partial is not complete")
    if data.get("models") != MODELS or sorted(data.get("cases", [])) != sorted(map(list, CASES)):
        raise ValueError("Pinned model revisions or required cases differ")
    env = data.get("environment", {})
    if any(env.get(k) != v for k, v in {"dtype": "float16", "transformers": "4.51.3",
                                        "attention_requested": "sdpa", "logits_to_keep": 1}.items()):
        raise ValueError("Protocol runtime settings differ")
    if not all(env.get(k) for k in ("python", "torch", "cuda", "gpu")):
        raise ValueError("Missing environment provenance")
    expected = {(model, *case, repeat) for model in MODELS for case in CASES for repeat in range(5)}
    seen = set()
    for row in data.get("results", []):
        key = tuple(row[k] for k in ("model", "batch", "input_tokens", "output_tokens", "repeat"))
        if key not in expected or key in seen:
            raise ValueError("Unexpected or duplicate model/shape/trial")
        seen.add(key)
        model, batch, prompt, output, repeat = key
        for metric in ("prefill_wall_ms", "prefill_event_ms", "decode_wall_ms", "decode_event_ms",
                       "decode_aggregate_tokens_per_s", "decode_mean_step_ms", "peak_allocated_bytes",
                       "peak_reserved_bytes", "prefill_peak_allocated_bytes"):
            value = row.get(metric)
            if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{metric} must be a finite positive measurement")
        if (row.get("decode_steps_per_sequence"), row.get("generated_tokens"),
            row.get("retained_tokens_per_sequence")) != (output - 1, batch * output, prompt + output - 1):
            raise ValueError("Output/step/KV accounting mismatch")
        if not math.isclose(row["decode_aggregate_tokens_per_s"], batch * (output - 1) * 1000 / row["decode_wall_ms"], rel_tol=1e-7):
            raise ValueError("Aggregate decode rate uses the wrong numerator or clock")
        if not math.isclose(row["decode_mean_step_ms"], row["decode_wall_ms"] / (output - 1), rel_tol=1e-7):
            raise ValueError("Mean step interval uses the wrong denominator")
        if row["peak_reserved_bytes"] < row["peak_allocated_bytes"] or row["peak_allocated_bytes"] < row["prefill_peak_allocated_bytes"]:
            raise ValueError("Impossible allocated/reserved peak ordering")
    if seen != expected:
        raise ValueError(f"Missing trials: found {len(seen)} of {len(expected)}")
    return len(seen)



print('Validated raw trials:', validate(RUN))


## 6. Focused profiles — two shapes, not the entire matrix

The 0.5B model, P=128 and P=4096, B=1, O=8. Each trace has named prefill and seven-step decode ranges. Open the local Chrome traces in a trace viewer; do not upload private artifacts to a public service. Record the visible gaps and dominant operators separately for each phase. Requested SDPA is not proof of a particular FlashAttention kernel: identify the actual selected operator/kernel from the trace.

Profiling changes execution. Keep these traces separate from unprofiled timings. They cannot explain the 1.5B/128-output case directly, establish HBM traffic, or validate a production p95. Hardware-counter diagnosis with Nsight is guided self-study, not required for this core artifact.


In [ ]:
PROFILE = focused_profiles(MODEL_PATHS[list(MODELS)[0]], EVIDENCE / 'profiles')
print(json.dumps(PROFILE, indent=2))


## 7. Required analysis and hand-in

Keep this lab in your **private** course repository; follow the current round's tag/deadline procedure in the student guide. Do not change the leaderboard server contract just to run a notebook.

Submit **topic02-analysis.md**, the unmodified core **topic02-benchmark.json**, **topic02-summary.json**, both **topic02-trace-p128.json / topic02-trace-p4096.json**, and **topic02-profile-summary.json**. No model weights, caches, secrets or executed notebook outputs belong in the public template.

Your analysis must contain:

1. A manifest: model revisions, GPU, driver, runtime, dtype, attention path, exact cases and timing boundaries. Add the driver from nvidia-smi to your report; the notebook records the other runtime fields.
2. Three predictions written before the run; observed medians **and min/max plus all five raw trials**. Five samples are not credible p95.
3. Two matched input/output contrasts for each model and the B1→B4 128/32 comparison. Aggregate rate uses B×(O−1); per-sequence mean cadence uses decode wall time/(O−1). Do not equate either with service capacity.
4. A KV calculation from model.config and P+O−1 retained tokens, reconciled with allocated/reserved/peak bytes. Explain at least two omitted memory terms.
5. A roofline estimate with declared precision, memory boundary, FLOP convention and **source for machine bandwidth/compute**. A worked hypothetical roof is allowed if labeled; do not fabricate hardware counters.
6. Two trace observations (phase + prompt length + operator/range), an alternative explanation, one failed or unresolved prediction, and the smallest next discriminating test. A screenshot without interpretation is insufficient.
7. Safety/reproducibility: partial failures if any, resource peaks/recovery, cache reuse and exact run identifier. Never hide an OOM or silently shrink the required matrix.

**Success criterion:** a reviewer can reproduce your arithmetic and understand why the evidence supports—or fails to support—your hypothesis. A faster number alone is not the learning outcome. Optional extensions must be labeled separately and must retain their own raw evidence.
